# 1.2 Hybrid Parsing with BeautifulSoup + Docling

**Author:** Daria

**Goal:** Extract and clean main text content from HTML news articles.

We implement a hybrid parsing strategy combining BeautifulSoup and Docling for robustness and quality.

In [ ]:
# Install libraries
!pip install beautifulsoup4 docling matplotlib


In [18]:
# Imports
from bs4 import BeautifulSoup
import pandas as pd
import os
import matplotlib.pyplot as plt
from docling.document_converter import DocumentConverter
import re

# Data Exploration

We begin by exploring the HTML structure based on four example articles, each representing a different folder:

- die-eth-karte-erhaelt-ein-neues-design.html
- erc-advanced-grants.html
- in-memory-of-konrad-steffen.html
- detecting-storms-thanks-to-gps.html

This will help us design parsing strategies that generalize across all four categories.

In [19]:
# Read a few example files
example_files = [
    '../HKNews/de_internal/2015/05/die-eth-karte-erhaelt-ein-neues-design.html',
    '../HKNews/de_news_events/2016/04/erc-advanced-grants.html',
    '../HKNews/en_internal/2020/08/in-memory-of-konrad-steffen.html',
    '../HKNews/en_news_events/2024/03/detecting-storms-thanks-to-gps.html'
]

examples = {}
for filepath in example_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        examples[os.path.basename(filepath)] = f.read()

# Display example names
list(examples.keys())

['die-eth-karte-erhaelt-ein-neues-design.html',
 'erc-advanced-grants.html',
 'in-memory-of-konrad-steffen.html',
 'detecting-storms-thanks-to-gps.html']

In [20]:
# Display the full raw HTML of all four example files

for name, html_content in examples.items():
    print(f"==== {name} ====")
    print(html_content)
    print("\n" + "="*150 + "\n")


==== die-eth-karte-erhaelt-ein-neues-design.html ====
<div class="text-image cq-dd-image">
<p>In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich.</p>
<p>Die Karte ist mit einer weiterentwickelten Version des bisher verwendeten RFID-Chips ausgestattet. Die elektronischen Funktionen der ETH-Karte und das Kartenmanagementsystem sind dieselben wie bisher. Ebenso bleiben die Informationen auf der ETH-Karte gleich: ETH-Logo, Mitarbeiterfoto, Name, Geburtsdatum, Gültigkeitsdauer, berufliche Rolle, organisationale Zuordnung, Identifikationsnummer und eventuell ASVZ-Berechtigung.</p>
<p>Der Austausch der alten durch die neue ETH-Karte beginnt ab Juli für Studierende und neu eintretende Mitarbeitende. ETH-Mitarbeitende, die bereits im Besitz einer ETH-Karte sind, erhalten die neue Karte per Post ab Mitte August. Sie müss

## Parsing with BeautifulSoup

We implement a BeautifulSoup-based parser that:
- Removes scripts and styles.
- Extracts section titles and paragraph text.
- Skips irrelevant sections (e.g., newsletter, download links).
- Preserves headings and paragraph separation


In [50]:
def parse_with_bs4(html_content):
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html_content, 'lxml')

    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Identify main article sections
    sections = soup.find_all('div', class_='text-image cq-dd-image')
    
    all_titles = []
    all_paragraphs = []

    footer_keywords = ['newsletter', 'staffnet', 'globe', 'download']

    for section in sections:
        # Preserve figcaption if it exists
        for fig in section.find_all('figure'):
            # Preserve figcaption if it exists
            figcaption = fig.find('figcaption')
            if figcaption:
                caption_text = figcaption.get_text(" ", strip=True)
                if caption_text:
                    all_paragraphs.append(caption_text)
            fig.decompose()


        # Skip footer sections based on title
        h2 = section.find('h2')
        if h2:
            title_text = h2.get_text(strip=True)
            if any(kw in title_text.lower() for kw in footer_keywords):
                continue
            all_titles.append(title_text)
        else:
            # If no h2, add "Main article" title for first/main content
            if not all_titles:
                all_titles.append("Main article")

        #  Paragraph extraction
        for p in section.find_all('p'):
            text = p.get_text(" ", strip=True)

            # Clean up link artifacts like "external page", "call_made"
            text = text.replace("external page", "").replace("call_made", "")
            text = ' '.join(text.split())  # Normalize extra spaces
            
            if not text:
                continue
            if any(kw in text.lower() for kw in footer_keywords):
                continue
            if len(text) < 20:
                continue
            all_paragraphs.append(text)

    return {
        'titles': all_titles,
        'body': '<br><br>'.join(all_paragraphs),  # use <br><br> for clearer formatting
        'paragraphs': all_paragraphs
    }


## Parsing with Docling

We set up a Docling `DocumentConverter` to automatically parse HTML files and export the content into clean markdown text.


In [51]:
# Create a Docling converter

docling_converter = DocumentConverter()

def parse_with_docling(filepath): 
    result = docling_converter.convert(filepath)
    text = result.document.export_to_markdown()
    return {
        'body': text
    }

---
## Implementing a Hybrid Parsing Approach

We now combine the strengths of both parsers to optimize extraction quality.  
- We use **BeautifulSoup** to extract clean, precise content.
- We use **Docling** to retrieve structural information (headings, markdown layout).
- The goal is to merge BeautifulSoup's clean body with Docling's section headers to produce RAG-ready chunks.


In [52]:
# Hybrid parser that works with in-memory HTML content
def hybrid_parser_from_content(filename, html_content):
    # Use BeautifulSoup to extract clean body and titles
    bs4_data = parse_with_bs4(html_content)

    # Save HTML to a temporary file to pass to Docling
    temp_path = f"/tmp/{filename}"
    with open(temp_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Use Docling to extract structured markdown
    docling_data = parse_with_docling(temp_path)

    return {
        'filename': filename,
        'bs4_body': bs4_data['body'],
        'bs4_titles': bs4_data['titles'],
        'bs4_paragraphs': bs4_data['paragraphs'],
        'docling_markdown': docling_data['body']
    }

## Applying the Hybrid Parser

We now apply our hybrid parser to the test HTML files.  
Each result includes:
- Cleaned text (from BeautifulSoup)
- Markdown structure (from Docling)
  
This prepares us for chunking and RAG-ready processing.


In [53]:
hybrid_results = []

for filename, html_content in examples.items():
    result = hybrid_parser_from_content(filename, html_content)
    hybrid_results.append(result)

In [54]:
# for printing the intermediate raw results

sample = hybrid_results[1]

print("=== Filename ===")
print(sample['filename'])
print("\n=== Cleaned BeautifulSoup Body ===\n")
print(sample['bs4_body'])  
print("\n=== Docling Markdown Output ===\n")
print(sample['docling_markdown'])  

=== Filename ===
erc-advanced-grants.html

=== Cleaned BeautifulSoup Body ===

Die ERC Advanced Grants gehören zu den begehrtesten Auszeichnungen im europäischen Forschungsraum. Mit ihnen fördert der Europäische Forschungsrat (ERC) ausschliesslich Projekte von etablierten Spitzenforschenden. Wer sich erfolgreich um diese Fördermittel bewirbt, erhält neben viel Renommee auch namhafte finanzielle Unterstützung. Die angenommenen Projekte werden während fünf Jahren mit rund 2,2 bis 3,8 Millionen Franken unterstützt.<br><br>17 Forschende der ETH Zürich haben sich für die ERC Advanced Grants beworben. Von ihnen schafften es 88 Prozent in die zweite Ausschreibungsrunde, und mehr als die Hälfte wurden mit «ausgezeichnet» (Kategorie A) bewertet und erfüllen somit die Kriterien für einen Grant. Wer am Schluss tatsächlich einen Grant erhält, hängt von vielen Faktoren ab, zum Beispiel davon, wie viel Geld insgesamt dem ERC zur Verfügung steht und wie viel davon an jeden einzelnen Forschenden geht.

In [55]:
def chunk_docling_markdown(markdown_text):
    """Split docling markdown into sections using ## headers.
    Fallback to one generic chunk if headers are missing or irrelevant.
    """
    chunks = []
    current = {"title": None, "text": ""}
    for line in markdown_text.splitlines():
        if line.startswith("## "):
            if current["title"] or current["text"].strip():
                chunks.append(current)
            current = {"title": line[3:].strip(), "text": ""}
        else:
            current["text"] += line + "\n"
    if current["title"] or current["text"].strip():
        chunks.append(current)

    # If only footers are present, ignore them and use a fallback chunk
    footer_keywords = ["staffnet", "newsletter", "kontakt", "about"]
    only_footers = all(any(kw in (c["title"] or "").lower() for kw in footer_keywords) for c in chunks)

    if len(chunks) < 2 or only_footers:
        return [{"title": "Main article", "text": ""}]
    else:
        return chunks


In [56]:
def distribute_bs4_text(paragraphs, docling_chunks):
    """Distribute list of BS4 paragraphs into Docling-defined chunks"""
    n = len(docling_chunks)
    m = len(paragraphs)
    avg = max(1, m // n) if n > 0 else m

    for i, chunk in enumerate(docling_chunks):
        start = i * avg
        end = m if i == n - 1 else (i + 1) * avg
        chunk["body"] = "\n\n" + "\n\n".join(paragraphs[start:end])
    
    return docling_chunks


In [57]:
# Apply hybrid merging
final_chunks_per_file = []

for r in hybrid_results:
    docling_chunks = chunk_docling_markdown(r['docling_markdown'])
    hybrid_chunks = distribute_bs4_text(r['bs4_paragraphs'], docling_chunks)

    
    final_chunks_per_file.append({
        'filename': r['filename'],
        'chunks': hybrid_chunks
    })


In [58]:
# Preview results from one file
for f in final_chunks_per_file:
    print(f"\n=== {f['filename']} ===")
    for c in f['chunks']:
        print(f"\n## {c['title']}\n{c['body']}")

    #break  # Remove this to loop over all files



=== die-eth-karte-erhaelt-ein-neues-design.html ===

## Main article


In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich.

Die Karte ist mit einer weiterentwickelten Version des bisher verwendeten RFID-Chips ausgestattet. Die elektronischen Funktionen der ETH-Karte und das Kartenmanagementsystem sind dieselben wie bisher. Ebenso bleiben die Informationen auf der ETH-Karte gleich: ETH-Logo, Mitarbeiterfoto, Name, Geburtsdatum, Gültigkeitsdauer, berufliche Rolle, organisationale Zuordnung, Identifikationsnummer und eventuell ASVZ-Berechtigung.

Der Austausch der alten durch die neue ETH-Karte beginnt ab Juli für Studierende und neu eintretende Mitarbeitende. ETH-Mitarbeitende, die bereits im Besitz einer ETH-Karte sind, erhalten die neue Karte per Post ab Mitte August. Sie müssen die neue Karte danach an einem 

In [59]:
output_dir = "parsed_markdown"
os.makedirs(output_dir, exist_ok=True)

for f in final_chunks_per_file:
    filename = f"{f['filename'].replace('.html', '')}.md"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, "w", encoding="utf-8") as out:
        out.write(f"# {f['filename']}\n\n")
        for c in f['chunks']:
            out.write(f"## {c['title']}\n\n")
            paragraphs = c['body'].split('\n\n')  # This must come from BS4 parsing!
            for p in paragraphs:
                clean = p.strip()
                if clean:
                    out.write(clean + "\n\n")  # Force paragraph block


